## 03 - EMG channel

Extracts the EMG channel (index 2), cleans it, builds the per-epoch EMG table, plots the EMG time course and attaches the EMG columns to an EEG epoch table.

Writes `cache/<animal>/csvs/*_raw_ch2.csv`, `cache/epochs_03/`, and under `outputs/03_emg_channel/` the EMG figure, `*_emg_epochs.csv`, `*_epoch_table_with_emg.csv` and `run_info.json`.

### Variables

In [ ]:
EDF_FOLDER = "../../Completed_Analyses/Emergence/raw_edfs"  # folder holding the .edf files (relative to this notebook)
EDF = "EmergRev_3Perc_Induction_PEEG02M_2026-08-27_15_49_57_export.edf"  # .edf file name inside EDF_FOLDER
ANIMAL_ID = "PEEG02M"  # animal id (label and cache sub-folder)
CONDITION = "Induction"  # condition name (label and raw CSV column)
RECORDING_NUMBER = 1  # session number (label)
EVENTS = {"onset": 1800}  # named events in recording-clock seconds; ONSET_EVENT must be among them
EEG_CHANNEL = 0  # EEG channel whose epoch table receives the EMG columns (step 6)
EMG_CHANNEL = 2  # EDF channel holding the EMG
ONSET_EVENT = "onset"  # event at 0 on the epoch time axis (None = recording clock); must match for EMG and EEG tables
EPOCH_S = 30  # epoch length (s); must be the same for the EMG and EEG tables
EMG_BAND = (20, 150)  # band-pass of the cleaned EMG (Hz)
WINDOW_S = 1.0  # RMS window of the envelope (s)
CLEAN = True  # True removes mains and other narrowband lines before the envelope; False = band-pass only
THRESHOLD_UV = None  # fixed epoch-RMS threshold (uV) for "moving"; None = the relative rule below
N_MAD = 3.0  # relative rule: threshold = median + N_MAD * 1.4826 * MAD of the epoch RMS values
BASELINE_WINDOW = None  # epoch-axis range (s) that defines median and MAD, e.g. (-1800, 0); None = whole recording
XLIM_MIN = None  # x range of the EMG figure in minutes, e.g. (-30, 60); None = automatic
FIGSIZE = (6, 2.8)  # EMG figure size in inches
FIT_RANGE = (1.0, 20.0)  # Hz range of the aperiodic fit in the EEG epoch table
OVERWRITE_RAW = False  # True re-extracts the raw CSVs
OVERWRITE_EPOCHS = False  # True refits the EEG epoch table even when the cache matches
WORK_DIR = "cache"  # processing cache (raw CSVs, epoch table), reused between runs
OUTPUT_DIR = "outputs/03_emg_channel"  # figures and tables; files are never overwritten
FIGURE_FORMATS = ("png", "svg")  # one file per format for each figure
DPI = 300  # resolution of the png

In [ ]:
from pathlib import Path  # path handling
import sys  # module search path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ecog").is_dir())  # repo root = first parent folder containing ecog/
sys.path.insert(0, str(ROOT))  # makes `import ecog` work without installing anything
import os  # path normalising
EDF_DIR = Path(os.path.abspath(EDF_FOLDER))  # absolute EDF folder with any `..` collapsed (relative paths start at this notebook's folder)
import mne  # EDF reader used by ecog
mne.set_log_level("WARNING")  # hides the EDF reader's progress messages
import contextlib, io, importlib  # output control and dynamic import
with contextlib.redirect_stderr(io.StringIO()):  # fooof prints a deprecation notice to stderr when it is first imported
    importlib.import_module("fooof")  # imported once here, before ecog needs it
import pandas as pd  # tables
from ecog import (Recording, extractRecording, extractEMG, emgEpochTable, emgEnvelope, figEMG, attachEMG, buildEpochTable,
                  saveFigure, saveTable, writeRunInfo)  # processing, EMG, epoch table, saving
WORK = Path(WORK_DIR)  # cache folder
OUT = Path(OUTPUT_DIR)  # output folder

### 1. Describe the recording

In [ ]:
# the recording's EEG channel is the default channel; the EMG channel is extracted separately
rec = Recording(animal_id=ANIMAL_ID, condition=CONDITION, recording_number=RECORDING_NUMBER, edf=EDF, edf_folder=EDF_DIR,
                channel_index=EEG_CHANNEL, events=EVENTS)
rec.validate()  # stops with one message if the file, channel or events are wrong
rec.label  # label used in every file name

### 2. Extract EEG and EMG channels to raw CSV (skipped when cached)

In [ ]:
extractRecording(rec, WORK, overwrite=OVERWRITE_RAW)  # EEG channel -> raw CSV
emgPath = extractEMG(rec, WORK, channel=EMG_CHANNEL, overwrite=OVERWRITE_RAW)  # EMG channel -> raw CSV with the same layout
emgPath  # file written

The raw EMG channel in these EDFs is dominated by 60 Hz mains and other narrowband lines (about 99% of 20-150 Hz power in the EmergRev files; the raw 1 s RMS is a flat ~345 uV regardless of behaviour).
`ecog.emg` therefore removes the lines before computing the envelope, reports them in `emg.attrs`, and flags moving epochs with a threshold relative to each recording (median + `N_MAD` x scaled MAD of the epoch RMS).

### 3. Per-epoch EMG table

In [ ]:
# one row per epoch: RMS (uV), power (dB) and a moving flag, on the same time grid as the EEG epoch table
emg = emgEpochTable(rec, WORK, channel=EMG_CHANNEL, onset_event=ONSET_EVENT, epoch_s=EPOCH_S, band=EMG_BAND, window_s=WINDOW_S,
                    threshold_uv=THRESHOLD_UV, n_mad=N_MAD, baseline_window=BASELINE_WINDOW, clean=CLEAN)
print(emg.attrs)  # threshold, window, raw_line_share, mains_removed_hz and lines_removed_hz of this recording
print("moving epochs:", int(emg["moving"].sum()), "of", len(emg))  # epochs above the threshold
emg.head()  # first rows

### 4. Raw versus cleaned envelope (the reason for the cleaning)

In [ ]:
def describeRms(rms):  # median and 5-95% range of an envelope (uV)
    s = pd.Series(rms)  # envelope as a series
    return f"median {s.median():.1f} uV, 5-95% range {s.quantile(.05):.1f}-{s.quantile(.95):.1f} uV"  # text summary


x = pd.read_csv(rec.rawCSV(WORK, EMG_CHANNEL)).iloc[:, 0].to_numpy(dtype=float)  # raw EMG trace in volts
print("raw (band-pass only):", describeRms(emgEnvelope(x, rec.fs, EMG_BAND, WINDOW_S, clean=False)[1]))  # flat, dominated by the lines
print("cleaned:", describeRms(emgEnvelope(x, rec.fs, EMG_BAND, WINDOW_S, clean=True)[1]))  # follows behaviour

### 5. EMG time course

In [ ]:
# 1 s RMS (thin), epoch means (thick), movement threshold (dashed) and moving epochs (shaded)
fig = figEMG(rec, emg, WORK, channel=EMG_CHANNEL, band=EMG_BAND, window_s=WINDOW_S, xlim_min=XLIM_MIN, figsize=FIGSIZE, clean=CLEAN)
saveFigure(fig, OUT / f"{rec.label}_ch{EMG_CHANNEL}_emg", dpi=DPI, formats=FIGURE_FORMATS)  # saved without overwriting

### 6. Attach EMG to the EEG epoch table

In [ ]:
# EEG epoch table of the same recording, same epoch length and onset event as the EMG table
table, _ = buildEpochTable({CONDITION: [(rec, EEG_CHANNEL)]}, WORK, cache=WORK / "epochs_03", force=OVERWRITE_EPOCHS,
                           onset_event=ONSET_EVENT, epoch_s=EPOCH_S, fit_range=FIT_RANGE)
table = attachEMG(table, emg)  # adds emg_rms_uv, emg_power_db, moving by session and epoch time; unmatched epochs get NaN
print("epochs without EMG:", int(table["emg_rms_uv"].isna().sum()))  # should be 0 when epoch settings agree
table.groupby("moving")[["exponent", "delta_rel", "spectral_entropy"]].agg(["count", "mean"]).round(3)  # EEG measures in still versus moving epochs

### 7. Save tables

In [ ]:
p1 = saveTable(emg, OUT / f"{rec.label}_ch{EMG_CHANNEL}_emg_epochs.csv")  # EMG epoch table
p2 = saveTable(table, OUT / f"{rec.label}_ch{EEG_CHANNEL}_epoch_table_with_emg.csv")  # EEG epoch table with EMG columns
info = writeRunInfo(OUT, settings=dict(edf=EDF, eeg_channel=EEG_CHANNEL, emg_channel=EMG_CHANNEL, band=EMG_BAND, window_s=WINDOW_S,
                                       clean=CLEAN, threshold_uv=emg.attrs["threshold_uv"], n_mad=N_MAD, epoch_s=EPOCH_S))  # versions and settings
print(p1, p2, info)  # files written